# 03 · Embeddings

Turns every kept review into a vector with `BAAI/bge-m3`: one model for all languages, so reviews about the same thing end up close together whatever language they're in. Vectors go to `data/embeddings/<model>/part-*.parquet`.

Re-run after collecting and cleaning more: only new reviews (and reviews edited since) get embedded. Too short reviews (cleaning flag `is_short`) are not embedded: they get no topic anyway. You can stop it at any time; finished parts are kept.

Why bge-m3: on 3,000 of our reviews it was the only model compared (vs Qwen3-Embedding-0.6B and multilingual-e5-base) whose clusters mixed languages instead of splitting by language. 

In [ ]:
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

from src.db import read
from src.embeddings import Embeddings, embed_new, load_encoder

In [ ]:
MODEL = "BAAI/bge-m3"
PROMPT = ""          # text put before every review; bge-m3 needs none
BATCH = 128          # lower it if CUDA runs out of memory
PART_SIZE = 20_000   # reviews per file, also the most a stop can lose

## Embed

Only kept reviews that are new or edited since they were embedded.

In [ ]:
encoder = load_encoder(MODEL)
embed_new(MODEL, encoder, PROMPT, BATCH, PART_SIZE)

## Delete vectors that are no longer needed (optional)

Older vectors of edited reviews, and vectors of reviews that are no longer kept or are too short (e.g. after a change to the cleaning rules). Only files that change are rewritten. Don't run it while something else embeds (`03`, `00_pipeline`).

In [ ]:
PRUNE = False
if PRUNE:
    needed = read.reviews(columns=["recommendation_id"], only_kept=True, skip_short=True, flag_columns=[])
    print(f"deleted {Embeddings(MODEL).prune(needed['recommendation_id']):,} vectors")

## Check: nearest reviews across languages

Each test sentence should find reviews about the same thing, in any language. Reads one file at a time, so memory stays small.

In [ ]:
emb = Embeddings(MODEL)
print(f"{len(emb.index):,} reviews in {len(emb.parts)} files")

In [ ]:
PROBES = [
    "Can't connect, the servers are always down.",
    "Не могу подключиться, серверы постоянно лежат.",
    "服务器老是连不上，根本进不去游戏。",
    "Way too expensive for what it is, wait for a sale.",
    "Слишком дорого за такой контент, ждите скидку.",
    "太贵了，不值这个价，等打折再买。",
    "Terrible optimization, constant FPS drops and stuttering.",
    "Ужасная оптимизация, постоянные просадки FPS и фризы.",
    "优化太差，掉帧卡顿严重。",
]
TOP = 5
probes = encoder.encode(PROBES, prompt=PROMPT, normalize_embeddings=True)
hits = []
for ids, vectors in tqdm(emb.vectors(), total=len(emb.parts), unit="file"):
    sims = vectors @ probes.T
    for j in range(len(PROBES)):
        for k in sims[:, j].argsort()[-TOP:]:
            hits.append((j, sims[k, j], ids[k]))
hits = pd.DataFrame(hits, columns=["probe", "similarity", "recommendation_id"])

info = read.reviews(ids=hits["recommendation_id"].unique(), columns=["recommendation_id", "review"],
                    flag_columns=["text_language"]).set_index("recommendation_id")
for j, probe in enumerate(PROBES):
    print(f"\n{probe}")
    for hit in hits[hits["probe"] == j].nlargest(TOP, "similarity").itertuples():
        text, lang = info.loc[hit.recommendation_id] if hit.recommendation_id in info.index else ("", "?")
        print(f"  {hit.similarity:.2f} [{lang}] {text[:100]!r}")